# Luxembourg School-to-School Distance Matrix: NetworkX and Pandana

This notebook is a compact, self-sufficient example for comparing routing backends in the reengineered `scalable_general_distances_per_country` project.

It does four things:

1. Resolves and downloads the Luxembourg OSM PBF if it is missing.
2. Extracts OSM `amenity=school` points from the PBF.
3. Loads the Luxembourg drivable road network and computes a school-to-school matrix capped at 20 km.
4. Computes the same matrix contract with NetworkX and, when available, Pandana, then compares the results.

The important design idea is that the data contract stays the same while the routing strategy changes. The downstream matrix consumer should not need to know whether NetworkX or Pandana produced the distances.

In [ ]:
from __future__ import annotations

from pathlib import Path
import sys
import time

import pandas as pd

# Locate the scalable package from this thesis notebook location.
REPO_ROOT = Path.cwd().resolve()
while REPO_ROOT.name != "Public-Infrastructure-Service-Access-remove-wfp" and REPO_ROOT.parent != REPO_ROOT:
    REPO_ROOT = REPO_ROOT.parent
if REPO_ROOT.name != "Public-Infrastructure-Service-Access-remove-wfp":
    # Fallback for a normal GitHub clone folder name.
    REPO_ROOT = Path.cwd().resolve()
    while not (REPO_ROOT / "Research-Sandbox" / "scalable_general_distances_per_country").exists() and REPO_ROOT.parent != REPO_ROOT:
        REPO_ROOT = REPO_ROOT.parent

SCALABLE_ROOT = REPO_ROOT / "Research-Sandbox" / "scalable_general_distances_per_country"
SRC_ROOT = SCALABLE_ROOT / "src"
assert (SRC_ROOT / "scalable_distances").exists(), f"Could not find scalable_distances under {SRC_ROOT}"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from scalable_distances.config import CountryDataSources
from scalable_distances.io import download_if_missing
from scalable_distances.network import load_driving_network
from scalable_distances.routing.strategies import NetworkXRouter, PandanaRouter

pd.set_option("display.max_columns", 30)
SCALABLE_ROOT

## Configuration

The cache location follows the same convention as the main distance pipeline. On a new machine, the PBF will be downloaded from Geofabrik into `C:/local/Download_Depot/luxembourg_data`. If that path is not appropriate, change `SHARED_CACHE_ROOT` before running the notebook.

In [ ]:
SHARED_CACHE_ROOT = Path(r"C:/local/Download_Depot")
COUNTRY_DIR = SHARED_CACHE_ROOT / "luxembourg_data"
OUTPUT_DIR = COUNTRY_DIR / "outputs" / "iana_backend_comparison"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MAX_DISTANCE_M = 20_000.0
RUN_PANDANA = True

sources = CountryDataSources(
    country_slug="luxembourg",
    iso3="LUX",
    base_dir=COUNTRY_DIR,
    geofabrik_region="europe",
    pbf_filename="luxembourg-latest.osm.pbf",
    worldpop_filename="lux_ppp_2020.tif",
)

pd.DataFrame([sources.as_manifest()])

## Download or Reuse Luxembourg OSM

The notebook is intentionally reproducible: it names the source URL and path explicitly, then uses `download_if_missing`, which reuses an existing file and only downloads when the cache is absent.

In [ ]:
pbf_path = sources.pbf_path
pbf_path = download_if_missing(sources.pbf_url, pbf_path)

pd.DataFrame([
    {
        "artifact": "luxembourg_osm_pbf",
        "url": sources.pbf_url,
        "path": str(pbf_path),
        "exists": pbf_path.exists(),
        "bytes": pbf_path.stat().st_size if pbf_path.exists() else None,
    }
])

## Extract Schools and Load the Road Network

For this teaching example, schools are taken directly from the Luxembourg PBF using OSM `amenity=school`. The helper below includes both school nodes and school ways/areas, using a simple centroid for ways. The same matrix pattern also works with richer geocoded school tables: they only need stable ids, longitude, and latitude columns.

In [ ]:
def extract_osm_school_points(pbf_path: Path) -> pd.DataFrame:
    """Extract OSM amenity=school nodes and ways as point records."""
    import osmium

    class SchoolHandler(osmium.SimpleHandler):
        def __init__(self) -> None:
            super().__init__()
            self.records = []

        def node(self, node) -> None:
            if node.tags.get("amenity") != "school" or not node.location.valid():
                return
            self.records.append(
                {
                    "school_id": f"osm_node_{int(node.id)}",
                    "osm_type": "node",
                    "osm_id": int(node.id),
                    "name": None if node.tags.get("name") is None else str(node.tags.get("name")),
                    "amenity": "school",
                    "lon": float(node.location.lon),
                    "lat": float(node.location.lat),
                }
            )

        def way(self, way) -> None:
            if way.tags.get("amenity") != "school":
                return
            coords = [(float(n.location.lon), float(n.location.lat)) for n in way.nodes if n.location.valid()]
            if not coords:
                return
            lon = sum(x for x, _ in coords) / len(coords)
            lat = sum(y for _, y in coords) / len(coords)
            self.records.append(
                {
                    "school_id": f"osm_way_{int(way.id)}",
                    "osm_type": "way",
                    "osm_id": int(way.id),
                    "name": None if way.tags.get("name") is None else str(way.tags.get("name")),
                    "amenity": "school",
                    "lon": lon,
                    "lat": lat,
                }
            )

    handler = SchoolHandler()
    handler.apply_file(str(pbf_path), locations=True)
    schools_df = pd.DataFrame(handler.records)
    if schools_df.empty:
        return schools_df
    return schools_df.drop_duplicates(subset=["school_id"]).reset_index(drop=True)


schools = extract_osm_school_points(pbf_path)
if schools.empty:
    raise RuntimeError("No OSM schools were extracted from the Luxembourg PBF.")

network = load_driving_network(pbf_path)

pd.DataFrame([
    {"object": "schools", "rows": len(schools)},
    {"object": "school_nodes", "rows": int(schools["osm_type"].eq("node").sum())},
    {"object": "school_ways", "rows": int(schools["osm_type"].eq("way").sum())},
    {"object": "network_nodes", "rows": len(network.nodes)},
    {"object": "network_edges", "rows": len(network.edges)},
])

## NetworkX Matrix Capped at 20 km

This is the portable reference path. It computes all school-to-school shortest paths up to the 20 km cutoff. The matrix is stored in the same long table style used elsewhere in the project: `source_id`, `target_id`, layer labels, road nodes, and distances.

In [ ]:
def prepare_school_points_for_router(router, schools_df: pd.DataFrame) -> pd.DataFrame:
    points = schools_df.rename(columns={"school_id": "source_id"}).copy()
    points["source_type"] = "school"
    snapped = router.snap(points[["source_id", "source_type", "lon", "lat", "name"]])
    return snapped


def networkx_school_matrix(network, schools_df: pd.DataFrame, *, cutoff_m: float) -> tuple[pd.DataFrame, pd.DataFrame, float]:
    import networkx as nx

    router = NetworkXRouter()
    router.prepare(network, {})
    snapped = prepare_school_points_for_router(router, schools_df)
    graph = router._graph
    rows = []
    start = time.perf_counter()
    for origin in snapped.itertuples(index=False):
        lengths = nx.single_source_dijkstra_path_length(
            graph,
            origin.nearest_node,
            cutoff=cutoff_m,
            weight="length_m",
        )
        for destination in snapped.itertuples(index=False):
            if origin.source_id == destination.source_id:
                continue
            distance = lengths.get(destination.nearest_node)
            if distance is None or distance > cutoff_m:
                continue
            rows.append(
                {
                    "source_id": origin.source_id,
                    "target_id": destination.source_id,
                    "source_type": "school",
                    "target_type": "school",
                    "source_node": origin.nearest_node,
                    "target_node": destination.nearest_node,
                    "network_dist": float(distance),
                    "total_dist": float(distance),
                    "router": "networkx",
                }
            )
    elapsed = time.perf_counter() - start
    return pd.DataFrame(rows), snapped, elapsed

networkx_matrix, networkx_snapped, networkx_seconds = networkx_school_matrix(network, schools, cutoff_m=MAX_DISTANCE_M)
networkx_path = OUTPUT_DIR / "luxembourg_school_to_school_networkx_20km.parquet"
networkx_matrix.to_parquet(networkx_path, index=False)

pd.DataFrame([
    {
        "router": "networkx",
        "schools": len(schools),
        "snapped_schools": len(networkx_snapped),
        "matrix_rows_under_20km": len(networkx_matrix),
        "runtime_seconds": networkx_seconds,
        "output": str(networkx_path),
    }
])

## Optional Pandana Matrix with the Same Contract

Pandana is optional because it can be sensitive to binary dependencies such as NumPy versions. The package imports it only if the Pandana strategy is selected. This cell tries Pandana and, if it is unavailable, records the reason and continues.

In [ ]:
def pandana_school_matrix(network, schools_df: pd.DataFrame, *, cutoff_m: float) -> tuple[pd.DataFrame | None, pd.DataFrame | None, float | None, str | None]:
    try:
        router = PandanaRouter()
        router.prepare(network, {})
        points = schools_df.rename(columns={"school_id": "source_id"}).copy()
        points["source_type"] = "school"
        snapped = router.snap(points[["source_id", "source_type", "lon", "lat", "name"]])
        destinations = snapped.rename(columns={"source_id": "target_id", "source_type": "target_type"})
        start = time.perf_counter()
        matrix = router.route_many(snapped, destinations)
        elapsed = time.perf_counter() - start
        matrix = matrix[matrix["source_id"].ne(matrix["target_id"])].copy()
        matrix = matrix[matrix["total_dist"].le(cutoff_m)].copy()
        matrix["router"] = "pandana"
        return matrix.reset_index(drop=True), snapped, elapsed, None
    except Exception as exc:
        return None, None, None, f"{type(exc).__name__}: {exc}"

if RUN_PANDANA:
    pandana_matrix, pandana_snapped, pandana_seconds, pandana_error = pandana_school_matrix(network, schools, cutoff_m=MAX_DISTANCE_M)
else:
    pandana_matrix, pandana_snapped, pandana_seconds, pandana_error = None, None, None, "RUN_PANDANA is False"

if pandana_matrix is not None:
    pandana_path = OUTPUT_DIR / "luxembourg_school_to_school_pandana_20km.parquet"
    pandana_matrix.to_parquet(pandana_path, index=False)
    pandana_status = pd.DataFrame([
        {
            "router": "pandana",
            "schools": len(schools),
            "snapped_schools": len(pandana_snapped),
            "matrix_rows_under_20km": len(pandana_matrix),
            "runtime_seconds": pandana_seconds,
            "output": str(pandana_path),
            "error": None,
        }
    ])
else:
    pandana_status = pd.DataFrame([
        {
            "router": "pandana",
            "schools": len(schools),
            "snapped_schools": None,
            "matrix_rows_under_20km": None,
            "runtime_seconds": None,
            "output": None,
            "error": pandana_error,
        }
    ])

pandana_status

## Compare the Two Backends

Both routers write the same columns. A practical comparison therefore joins on the origin school and destination school and checks row counts and distance deltas. Small numerical differences are normal because engines can differ in graph construction and nearest-node handling, but large differences should be investigated.

In [ ]:
def compare_backend_matrices(networkx_df: pd.DataFrame, pandana_df: pd.DataFrame | None) -> pd.DataFrame:
    if pandana_df is None:
        return pd.DataFrame([{"comparison": "pandana unavailable", "value": pandana_error}])
    nx_small = networkx_df[["source_id", "target_id", "total_dist"]].rename(columns={"total_dist": "networkx_dist_m"})
    pdna_small = pandana_df[["source_id", "target_id", "total_dist"]].rename(columns={"total_dist": "pandana_dist_m"})
    joined = nx_small.merge(pdna_small, on=["source_id", "target_id"], how="outer", indicator=True)
    matched = joined[joined["_merge"].eq("both")].copy()
    if not matched.empty:
        matched["abs_delta_m"] = (matched["networkx_dist_m"] - matched["pandana_dist_m"]).abs()
    return pd.DataFrame([
        {"comparison": "networkx rows", "value": len(networkx_df)},
        {"comparison": "pandana rows", "value": len(pandana_df)},
        {"comparison": "matched pairs", "value": int(joined["_merge"].eq("both").sum())},
        {"comparison": "networkx only pairs", "value": int(joined["_merge"].eq("left_only").sum())},
        {"comparison": "pandana only pairs", "value": int(joined["_merge"].eq("right_only").sum())},
        {"comparison": "mean absolute delta m", "value": None if matched.empty else float(matched["abs_delta_m"].mean())},
        {"comparison": "max absolute delta m", "value": None if matched.empty else float(matched["abs_delta_m"].max())},
    ])

comparison = compare_backend_matrices(networkx_matrix, pandana_matrix)
comparison

## How to Extend

To add another routing backend, implement the same three-method strategy interface:

- `prepare(network, config)`: build backend-specific network state.
- `snap(points)`: add a `nearest_node` column to points.
- `route_many(origins, destinations)`: return the long-form matrix table.

If that contract is respected, the downstream matrix comparison, persistence, facility-location, and TSP code can stay unchanged.

In [ ]:
backend_summary = pd.concat(
    [
        pd.DataFrame([
            {
                "router": "networkx",
                "schools": len(schools),
                "snapped_schools": len(networkx_snapped),
                "matrix_rows_under_20km": len(networkx_matrix),
                "runtime_seconds": networkx_seconds,
                "output": str(networkx_path),
                "error": None,
            }
        ]),
        pandana_status,
    ],
    ignore_index=True,
)
backend_summary